# Project Delay Early Warning System
## Exploratory Data Analysis & Model Evaluation

This notebook covers the machine learning workflow for predicting IT project delays 4 weeks before they happen. 

### Workflow Outline:
1. **Load Data**: Load simulated raw project records.
2. **Exploratory Analysis**: Visualize distributions and correlation patterns.
3. **Feature Engineering**: Develop derived features representing risk composite scores.
4. **Model Training**: Fit a Random Forest Classifier with class-balancing.
5. **Evaluation**: Perform ROC-AUC testing, 5-Fold Cross Validation, and inspect Feature Importances.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Ensure charts show up inline
%matplotlib inline
sns.set_theme(style="whitegrid")

### 1. Load Data
Let's load the raw project simulator dataset from `data/projects.csv`.

In [ ]:
# Load projects
df = pd.read_csv('../data/projects.csv')
print(f"Dataset shape: {df.shape}")
df.head()

### 2. Exploratory Data Analysis
Let's check the target variable balance. How many projects are delayed vs. on time?

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='delayed', palette=['#3B9C6E', '#E24B4A'])
ax.set_title("Distribution of Delayed Projects")
ax.set_xticklabels(["On Time (0)", "Delayed (1)"])
plt.ylabel("Count of Projects")
plt.show()

#### Feature Correlations
Let's check how the raw numerical variables relate to each other and to the target `delayed` variable.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of Raw Features")
plt.show()

### 3. Feature Engineering
Now we'll define a set of derived features to capture compound risks, capacity constraints, and gaps in schedules.

In [ ]:
def engineer_features(data):
    df_eng = data.copy()
    # Velocity trend: work completed relative to elapsed time
    df_eng['velocity_trend'] = df_eng['tasks_completed_pct'] / df_eng['days_elapsed'].replace(0, 1)
    # Scope Risk: interaction between scope changes and outstanding bugs
    df_eng['scope_risk'] = df_eng['scope_changes'] * df_eng['bugs_open']
    # Team pressure: tasks completed relative to team availability
    df_eng['team_pressure'] = df_eng['tasks_completed_pct'] / df_eng['team_availability_pct'].replace(0, 1)
    # Completion gap: percentage work completed minus percentage time elapsed
    df_eng['completion_gap'] = (df_eng['tasks_completed_pct'] / 100) - (df_eng['days_elapsed'] / df_eng['planned_duration_days'].replace(0, 1))
    # Bug density: open bugs relative to team size
    df_eng['bug_density'] = df_eng['bugs_open'] / df_eng['team_size'].replace(0, 1)
    return df_eng

df_eng = engineer_features(df)
df_eng.head()

#### Visualize Engineered Features
Let's look at `completion_gap` (which measures if a project is falling behind) compared to the actual delay outcome.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_eng, x='delayed', y='completion_gap', palette=['#3B9C6E', '#E24B4A'])
plt.title("Completion Gap by Project Delay Outcome")
plt.xticks([0, 1], ["On Time", "Delayed"])
plt.ylabel("Completion Gap (negative = behind schedule)")
plt.show()

### 4. Model Training & Evaluation
We select the engineered columns for training a Random Forest Classifier model.

In [ ]:
FEATURE_COLS = [
    'team_size',
    'sprint_velocity',
    'tasks_completed_pct',
    'scope_changes',
    'bugs_open',
    'team_availability_pct',
    'velocity_trend',
    'scope_risk',
    'completion_gap',
    'bug_density'
]

X = df_eng[FEATURE_COLS]
y = df_eng['delayed']

# 80/20 train/test split with stratification on target labels
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size:  {X_test.shape[0]}")

In [ ]:
# Fit Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=4,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)

#### Evaluation Metrics

In [ ]:
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, probs)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print(f"ROC-AUC Score on Test Set:    {roc_auc:.4f}")
print(f"ROC-AUC 5-Fold Cross Val:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))

# Confusion Matrix Plot
plt.figure(figsize=(5, 4))
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted On-Time", "Predicted Delayed"],
            yticklabels=["Actual On-Time", "Actual Delayed"])
plt.title("Confusion Matrix")
plt.ylabel("Actual Outcome")
plt.xlabel("Model Prediction")
plt.show()

### 5. Feature Importances
Let's see what features drive the predictions most heavily.

In [ ]:
fi = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(8, 5))
sns.barplot(data=fi, x='importance', y='feature', palette="viridis")
plt.title("Feature Importance breakdown - Random Forest Model")
plt.xlabel("Relative Importance Score")
plt.ylabel("Feature")
plt.show()